# HPC Version

Use kernel: `Python (cellpose-infer)`.



In [1]:
import socket, torch
print("host:", socket.gethostname())
print("cuda:", torch.cuda.is_available())
print("device_count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


host: hpctpa3pc0001
cuda: True
device_count: 1
gpu: NVIDIA A30


## Cellpose-SAM: superhuman generalization for cellular segmentation

Marius Pachitariu, Michael Rariden, Carsen Stringer

[paper](https://www.biorxiv.org/content/10.1101/2025.04.28.651001v1) | [code](https://github.com/MouseLand/cellpose)

This notebook shows how to process your own 2D or 3D images, saved on Google Drive.

This notebook is adapted from the notebook by Pradeep Rajasekhar, inspired by the [ZeroCostDL4Mic notebook series](https://github.com/HenriquesLab/ZeroCostDL4Mic/wiki).

### Install Cellpose-SAM


Check GPU and instantiate model - will download weights.

In [2]:
import numpy as np
from cellpose import models, core, io, plot
from pathlib import Path
import matplotlib.pyplot as plt

import cv2
from scipy.ndimage import find_objects
from matplotlib.colors import hsv_to_rgb

from natsort import natsorted
import random
import warnings
from tqdm import trange, tqdm


io.logger_setup() # run this to get printing of progress

#Check if colab notebook instance has GPU access
if core.use_gpu()==False:
  raise ImportError("No GPU access, change your runtime")
from concurrent.futures import ProcessPoolExecutor, wait, FIRST_COMPLETED
import os

import time


ModuleNotFoundError: No module named 'cellpose'

Input directory with your images:

In [ ]:
# # *** change to your google drive folder path ***
# dir = "/share/lab_soupir/IHC/ideas/HnE_segmentation/data/outputs/RCC3/preprocessing/output/RCC_3.svs"
# dir = Path(dir)
# if not dir.exists():
#   raise FileNotFoundError("directory does not exist")

# # *** change to your image extension ***
# image_ext = ".png"

# # list all files
# files = natsorted([f for f in dir.glob("*"+image_ext) if "_masks" not in f.name and "_flows" not in f.name])

# if(len(files)==0):
#   raise FileNotFoundError("no image files found, did you specify the correct folder and extension?")
# else:
#   print(f"{len(files)} images in folder:")

# for f in files:
#   print(f.name)


## Run Cellpose-SAM on one image in folder

Here are some of the parameters you can change:

* ***flow_threshold*** is  the  maximum  allowed  error  of  the  flows  for  each  mask.   The  default  is 0.4.
    *  **Increase** this threshold if cellpose is not returning as many masks as you’d expect (or turn off completely with 0.0)
    *   **Decrease** this threshold if cellpose is returning too many ill-shaped masks.

* ***cellprob_threshold*** determines proability that a detected object is a cell.   The  default  is 0.0.
    *   **Decrease** this threshold if cellpose is not returning as many masks as you’d expect or if masks are too small
    *   **Increase** this threshold if cellpose is returning too many masks esp from dull/dim areas.

* ***tile_norm_blocksize*** determines the size of blocks used for normalizing the image. The default is 0, which means the entire image is normalized together.
  You may want to change this to 100-200 pixels if you have very inhomogeneous brightness across your image.



In [ ]:
# selected_channels = []
# for i, c in enumerate([first_channel, second_channel, third_channel]):
#   if c == 'None':
#     continue
#   if int(c) > img.shape[-1]:
#     assert False, 'invalid channel index, must have index greater or equal to the number of channels'
#   if c != 'None':
#     selected_channels.append(int(c))



# img_selected_channels = np.zeros_like(img)
# img_selected_channels[:, :, :len(selected_channels)] = img[:, :, selected_channels]


# flow_threshold = 0.4
# cellprob_threshold = 0.0
# tile_norm_blocksize = 0

# masks, flows, styles = model.eval(img_selected_channels, batch_size=32, flow_threshold=flow_threshold, cellprob_threshold=cellprob_threshold,
#                                   normalize={"tile_norm_blocksize": tile_norm_blocksize})

# fig = plt.figure(figsize=(12,5))
# plot.show_segmentation(fig, img_selected_channels, masks, flows[0])
# plt.tight_layout()
# plt.show()


## Run Cellpose-SAM on folder of images

if you have many large images, you may want to run them as a loop over images



In [ ]:
# masks_ext = ".png" if image_ext == ".png" else ".tif"
# for i in trange(len(files)):
#     f = files[i]
#     img = io.imread(f)
#     masks, flows, styles = model.eval(img, batch_size=32, flow_threshold=flow_threshold, cellprob_threshold=cellprob_threshold,
#                                   normalize={"tile_norm_blocksize": tile_norm_blocksize})
#     io.imsave(dir / (f.stem + "_masks" + masks_ext), masks)

if you have small images, you may want to load all of them first and then run, so that they can be batched together on the GPU

In [ ]:
# masks_ext = ".png" if image_ext == ".png" else ".tif"

# print("loading images")
# imgs = [io.imread(files[i]) for i in trange(len(files))]

# print("running cellpose-SAM")
# masks, flows, styles = model.eval(imgs, batch_size=10, flow_threshold=flow_threshold, cellprob_threshold=cellprob_threshold,
#                                   normalize={"tile_norm_blocksize": tile_norm_blocksize})

# # print("saving masks")
# for i in trange(len(files)):
#     f = files[i]
#     io.imsave(dir / (f.stem + "_masks" + masks_ext), masks[i])

In [ ]:
# for i in range(len(files)):
#   print("Image",i)
#   print(len(np.unique(masks[i])))
#   fig = plt.figure(figsize=(12,5))
#   plot.show_segmentation(fig,imgs[i],maski=masks[i],flowi=flows[i][0])
#   plt.tight_layout()
#   plt.show()

to save your masks for ImageJ, run the following code:

In [ ]:
# for i in trange(len(files)):
#     f = files[i]
#     masks0 = io.imsave(dir / (f.name + "_masks" + masks_ext))
#     io.save_rois(masks0, f)

In [12]:
input_dir= Path("/share/lab_soupir/IHC/ideas/HnE_segmentation/data/outputs/RCC5/preprocessing/output/RCC_5.svs")
out_dir = Path("/share/lab_teng/trainee/tusharsingh/cell-seg/inference/cellpose/outputs/RCC_5")

In [13]:
flow_threshold = 0.4
cellprob_threshold = 0.0
tile_norm_blocksize = 0

In [14]:
# ---------------------------------------------------
# MASK OVERLAY (Cellpose logic)
# ---------------------------------------------------

def mask_overlay(img, masks, colors=None):
    """
    Overlay instance masks on an image using HSV coloring.

    Parameters
    ----------
    img : ndarray
        Image [H,W] or [H,W,C]

    masks : ndarray
        Instance mask [H,W]
        0 = background
        1..N = object IDs

    colors : optional ndarray [N,3]
        RGB colors for objects

    Returns
    -------
    RGB image with colored segmentation overlay
    """

    # convert to grayscale brightness
    if img.ndim > 2:
        img_gray = img.astype(np.float32).mean(axis=-1)
    else:
        img_gray = img.astype(np.float32)

    H, W = img_gray.shape

    HSV = np.zeros((H, W, 3), np.float32)

    # brightness channel
    if img_gray.max() > 1:
        val = img_gray / 255.0
    else:
        val = img_gray

    HSV[:, :, 2] = np.clip(val * 1.5, 0, 1)

    # number of instances
    n_masks = int(masks.max())

    if n_masks == 0:
        return (hsv_to_rgb(HSV) * 255).astype(np.uint8)

    # generate random hues
    hues = np.linspace(0, 1, n_masks + 1)[np.random.permutation(n_masks)]

    for n in range(n_masks):

        coords = np.where(masks == n + 1)

        if coords[0].size == 0:
            continue

        if colors is None:
            HSV[coords[0], coords[1], 0] = hues[n]
        else:
            HSV[coords[0], coords[1], 0] = colors[n, 0]

        HSV[coords[0], coords[1], 1] = 1.0

    RGB = (hsv_to_rgb(HSV) * 255).astype(np.uint8)

    return RGB


# ---------------------------------------------------
# MASK OUTLINES (exact object contours)
# ---------------------------------------------------

def masks_to_outlines(masks):
    """
    Compute object outlines from instance mask.

    Works for both 2D and 3D masks.
    """

    if masks.ndim > 3 or masks.ndim < 2:
        raise ValueError("masks_to_outlines expects 2D or 3D array")

    outlines = np.zeros(masks.shape, dtype=bool)

    # handle 3D stacks
    if masks.ndim == 3:
        for i in range(masks.shape[0]):
            outlines[i] = masks_to_outlines(masks[i])
        return outlines

    # bounding boxes per object
    slices = find_objects(masks.astype(int))

    for i, slc in enumerate(slices):

        if slc is None:
            continue

        sr, sc = slc

        region = (masks[sr, sc] == (i + 1)).astype(np.uint8)

        contours, _ = cv2.findContours(
            region,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_NONE
        )

        if len(contours) == 0:
            continue

        pts = np.concatenate(contours, axis=0).squeeze()

        vc = pts[:, 0] + sc.start
        vr = pts[:, 1] + sr.start

        outlines[vr, vc] = True

    return outlines


# ---------------------------------------------------
# DRAW OUTLINES ON IMAGE
# ---------------------------------------------------

def draw_outlines(image, outlines):
    """
    Draw red boundaries on image.
    """

    img = image.copy()

    if img.ndim == 2:
        img = np.stack([img]*3, axis=-1)

    if img.max() <= 1:
        img = (img * 255).astype(np.uint8)

    y, x = np.nonzero(outlines)

    img[y, x] = [255, 0, 0]

    return img


# ---------------------------------------------------
# MAIN VISUALIZATION FUNCTION
# ---------------------------------------------------

def save_segmentation(img, masks, out_dir, name,ncells):
    """
    Save segmentation results to disk instead of displaying them.

    Panels saved:
    1. original image
    2. outlines
    3. colored masks
    """

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    outlines = masks_to_outlines(masks)
    overlay = mask_overlay(img, masks)
    outline_img = draw_outlines(img, outlines)

    # Fast PNG writer (same panel logic, no matplotlib rendering)
    def _to_rgb_uint8(x):
        arr = np.asarray(x)
        if arr.ndim == 2:
            arr = np.stack([arr] * 3, axis=-1)
        if arr.dtype != np.uint8:
            arr = arr.astype(np.float32)
            if arr.max() <= 1:
                arr = arr * 255.0
            arr = np.clip(arr, 0, 255).astype(np.uint8)
        return arr

    img_rgb = _to_rgb_uint8(img)
    outline_rgb = _to_rgb_uint8(outline_img)
    overlay_rgb = _to_rgb_uint8(overlay)

    panel = np.concatenate([img_rgb, outline_rgb, overlay_rgb], axis=1)
    save_path = out_dir / f"{name}_{ncells}.png"

    cv2.imwrite(
        str(save_path),
        cv2.cvtColor(panel, cv2.COLOR_RGB2BGR),
        [int(cv2.IMWRITE_PNG_COMPRESSION), 1],
    )


In [15]:
# # *** change to your google drive folder path ***
# dir = "inference/cellpose/inputs/CMU-1-Small-Region"
# dir = Path(dir)
# if not dir.exists():
#   raise FileNotFoundError("directory does not exist")

# # *** change to your image extension ***
# image_ext = ".png"

# # list all files
# files = natsorted([f for f in dir.glob("*"+image_ext) if "_masks" not in f.name and "_flows" not in f.name])

# if(len(files)==0):
#   raise FileNotFoundError("no image files found, did you specify the correct folder and extension?")
# else:
#   print(f"{len(files)} images in folder:")

# for f in files:
#   print(f.name)

# masks_ext = ".png" if image_ext == ".png" else ".tif"

# print("loading images")
# imgs = [io.imread(files[i]) for i in trange(len(files))]

# model = models.CellposeModel(gpu=True)
# print("running cellpose-SAM")
# masks, flows, styles = model.eval(imgs, batch_size=10, flow_threshold=flow_threshold, cellprob_threshold=cellprob_threshold,
#                                   normalize={"tile_norm_blocksize": tile_norm_blocksize})

# for i in range(len(files)):
#   print("Image",i)
#   ncells=len(np.unique(masks[i]))
#   fig = plt.figure(figsize=(12,5))
#   # plot.show_segmentation(fig,imgs[i],maski=masks[i],flowi=flows[i][0])
#   save_segmentation(imgs[i], masks[i], out_dir, name=files[i].stem,ncells=ncells)
#   plt.tight_layout()
#   plt.show()


In [16]:
# ---------------------------
# Settings
# ---------------------------

image_ext = ".png"
n_samples = 2         # set to None to use all files
random_seed = 42
show_file_names = False  # True if you want to print all selected file names

# optional: suppress noisy warnings if needed
# warnings.filterwarnings(
#     "ignore",
#     category=FutureWarning,
#     message=".*torch.load.*weights_only.*"
# )

# ---------------------------
# Validate input directory
# ---------------------------

if not input_dir.exists():
    raise FileNotFoundError(f"directory does not exist: {input_dir}")

# ---------------------------
# Collect files
# ---------------------------

files = natsorted([
    f for f in input_dir.glob("*" + image_ext)
    if "_masks" not in f.name and "_flows" not in f.name
])

if len(files) == 0:
    raise FileNotFoundError(
        "no image files found, did you specify the correct folder and extension?"
    )

# ---------------------------
# Random subset selection
# ---------------------------

if n_samples is not None:
    random.seed(random_seed)
    files = random.sample(files, max(n_samples, len(files))) #min/max

print(f"Processing {len(files)} image(s) from folder: {input_dir}")

if show_file_names:
    for f in files:
        print(f.name)

masks_ext = ".png" if image_ext == ".png" else ".tif"

# ---------------------------
# Load images
# ---------------------------

print("Loading images...")
imgs = [io.imread(files[i]) for i in trange(len(files))]

# ---------------------------
# Run Cellpose-SAM
# ---------------------------

print("Loading Cellpose model...")
model = models.CellposeModel(gpu=True)

print("Running Cellpose-SAM inference...")
masks, flows, styles = model.eval(
    imgs,
    batch_size=32,
    flow_threshold=flow_threshold,
    cellprob_threshold=cellprob_threshold,
    normalize={"tile_norm_blocksize": tile_norm_blocksize}
)

# ---------------------------
# Save outputs (CPU multiprocessing)
# ---------------------------

print("Saving segmentation outputs...")
write_t0 = time.perf_counter()

WRITE_WITH_MULTIPROCESSING = True
WRITE_WORKERS = max(1, min(12, (os.cpu_count() or 1) // 4 or 1))
print(f"Write multiprocessing: {WRITE_WITH_MULTIPROCESSING}, workers={WRITE_WORKERS}")

if WRITE_WITH_MULTIPROCESSING:
    MAX_PENDING_WRITE_JOBS = max(WRITE_WORKERS * 4, 8)

    with ProcessPoolExecutor(max_workers=WRITE_WORKERS) as ex:
        pending = {}
        saved_count = 0

        for i, f in enumerate(files):
            ncells = len(np.unique(masks[i]))
            fut = ex.submit(save_segmentation, imgs[i], masks[i], str(out_dir), f.stem, ncells)
            pending[fut] = f.name

            if len(pending) >= MAX_PENDING_WRITE_JOBS:
                done, _ = wait(pending, return_when=FIRST_COMPLETED)
                for d in done:
                    fname = pending.pop(d)
                    try:
                        d.result()
                    except Exception as e:
                        print(f"[WRITE-ERROR] {fname} -> {e}")
                    saved_count += 1
                    if saved_count % 10 == 0:
                        print(f"Saved {saved_count}/{len(files)}")

        # flush remaining writes
        for d, fname in list(pending.items()):
            try:
                d.result()
            except Exception as e:
                print(f"[WRITE-ERROR] {fname} -> {e}")
            saved_count += 1
            if saved_count % 10 == 0:
                print(f"Saved {saved_count}/{len(files)}")
else:
    for i, f in enumerate(files):
        ncells = len(np.unique(masks[i]))
        save_segmentation(
            imgs[i],
            masks[i],
            out_dir,
            name=f.stem,
            ncells=ncells
        )

        if i % 10 == 0:
            print(f"Saved {i+1}/{len(files)}")

write_elapsed = time.perf_counter() - write_t0
if write_elapsed > 0:
    print(f"Write stage wall time: {write_elapsed:.2f}s | throughput: {len(files)/write_elapsed:.2f} img/s")


Processing 2208 image(s) from folder: /share/lab_soupir/IHC/ideas/HnE_segmentation/data/outputs/RCC5/preprocessing/output/RCC_5.svs
Loading images...


100%|██████████████████████████████████████████████████████████████████████████████| 2208/2208 [00:17<00:00, 128.39it/s]

Loading Cellpose model...
2026-03-14 18:48:47,100 [INFO] ** TORCH CUDA version installed and working. **


2026-03-14 18:48:47,101 [INFO] >>>> using GPU (CUDA)
2026-03-14 18:48:48,481 [INFO] >>>> loading model /home/80033718/.cellpose/models/cpsam
Running Cellpose-SAM inference...
2026-03-14 18:48:48,885 [INFO] 0%|          | 0/2208 [00:00<?, ?it/s]
2026-03-14 18:49:03,695 [WARNING] no seeds found in get_masks_torch - no masks found.
2026-03-14 18:49:03,959 [INFO] No cell pixels found.
2026-03-14 18:49:13,536 [INFO] No cell pixels found.
2026-03-14 18:49:14,992 [INFO] No cell pixels found.
2026-03-14 18:49:16,585 [INFO] No cell pixels found.
2026-03-14 18:49:18,639 [INFO] No cell pixels found.
2026-03-14 18:49:18,940 [INFO] 9%|9         | 203/2208 [00:30<04:56,  6.75it/s]
2026-03-14 18:49:19,629 [INFO] No cell pixels found.
2026-03-14 18:49:20,782 [INFO] No cell pixels found.
2026-03-14 18:49:31,347 [INFO] No cell pixels found.
2026-03-14 18:49:35,620 [WARNING] no seeds found in get_masks_torch - no masks found.
2026-03-14 18:49:48,987 [INFO] 18%|#8        | 406/2208 [01:00<04:26,  6.76it/s